### 1. Projection du Taux de Pauvreté en 2021 à l'Aide d'un Modèle de Régression Logistique Entraîné sur les Données de 2018

- Ce script est un outil puissant pour anticiper l'évolution de la pauvreté en exploitant l’imagerie satellitaire et le machine learning.
- Il permet de faire une projection du taux de pauvreté en 2021 sans nécessiter de nouvelles enquêtes de terrain.
- Il garantit la cohérence des transformations appliquées aux données de 2021, en réutilisant les mêmes outils de prétraitement (scaling et PCA) que ceux utilisés pour 2018.
- Il fournit un fichier exploitable contenant les estimations de la pauvreté en 2021 pour des analyses futures ou une validation par des experts.

Ce script applique un modèle de régression logistique pré-entraîné sur les données de 2018 pour prédire les classes binaires (pcexp_binaire) des observations de 2021. Il est conçu pour projeter le taux de pauvreté en 2021 sans disposer des valeurs réelles de la variable cible.

Étapes du script :
1. Chargement des données de 2021
   - Lecture du fichier CSV contenant les caractéristiques extraites des images satellites de 2021.
2. Prétraitement des données de 2021
   - Vérification de la présence des colonnes de caractéristiques (feature_0 à feature_4095).
   - Normalisation des données en utilisant le même StandardScaler que celui appliqué aux données de 2018.
   - Réduction de dimension via le même modèle PCA utilisé en 2018.
3. Chargement du Modèle de Régression Logistique (2018)
   - Le modèle entraîné en 2018 est chargé à partir d’un fichier .pkl.
4. Prédictions sur les données de 2021
   - Le modèle est appliqué aux données de 2021 transformées (après normalisation et PCA).
   - Une nouvelle colonne pcexp_binaire_pred est ajoutée au fichier de 2021 avec les classes prédites (0 ou 1).
5. Sauvegarde des Prédictions
   - Le fichier mis à jour avec les prédictions est sauvegardé dans un nouveau CSV pour analyses ultérieures.
6. calcul du taux de pauvreté prévisionnel pour 2021

In [ ]:
import pandas as pd
import os
import pickle

# Définition des chemins
input_csv_2021 = r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConv_sans_augm_couche_gelee_batch_16_base_EHCVM_2018_pour_an_2021_version_2.csv" #Pour 2023 remplacer ce fichier
model_path = r"D:\wealth_predict_sentinel\models\fullyConv_sans_augm_couche_gelee_batch_16_logistic_pca_5-fold_logistic_reg_version_2.pkl"
scaler_path = r"D:\wealth_predict_sentinel\models\scaler_fullyConv_sans_augm_couche_gelee_batch_16_cross_validation_version_2.pkl"
pca_path = r"D:\wealth_predict_sentinel\models\pca_fullyConv_sans_augm_couche_gelee_batch_16_cross_validation_version_2.pkl"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"

# Chargement des données de 2021
df_2021 = pd.read_csv(input_csv_2021)

# Vérification des colonnes nécessaires
features = [f'feature_{i}' for i in range(4096)]
required_columns = features + ['hhweight', 'hhsize']  # Ajout des colonnes nécessaires au calcul du taux de pauvreté

missing_columns = [col for col in required_columns if col not in df_2021.columns]
if missing_columns:
    raise ValueError(f"Les colonnes suivantes sont manquantes dans les données de 2021 : {missing_columns}")

X_2021 = df_2021[features]

# Chargement du scaler et du PCA utilisés en 2018
with open(scaler_path, 'rb') as file:
    scaler = pickle.load(file)

with open(pca_path, 'rb') as file:
    pca = pickle.load(file)

# Normalisation des données 2021 avec le même scaler
X_2021_scaled = scaler.transform(X_2021)

# Réduction de dimension avec le même PCA
X_2021_pca = pca.transform(X_2021_scaled)

# Chargement du modèle entraîné en 2018
# La ligne suivante charge le modèle entraîné avec ses hyperparamètres, y compris class_weight='balanced'
with open(model_path, 'rb') as file:
    logistic_model = pickle.load(file)

# Prédictions sur les données de 2021
y_pred_2021 = logistic_model.predict(X_2021_pca)

# Ajout des prédictions au DataFrame
df_2021['pcexp_binaire_pred'] = y_pred_2021

# Calcul du taux de pauvreté prévisionnel pour 2021 en utilisant les pondérations de 2018
weighted_predicted_poverty_2021 = (
    (df_2021['hhweight'] * df_2021['hhsize'] * df_2021['pcexp_binaire_pred']).sum() /
    (df_2021['hhweight'] * df_2021['hhsize']).sum()
)

print(f"Taux de pauvreté prévisionnel en 2021 : {weighted_predicted_poverty_2021:.4%}")

# Sauvegarde des résultats finaux
final_csv_path = os.path.join(output_dir_csv, "fullyConv_sans_augm_couche_gelee_batch_16_test_with_poverty_2021_logistic_reg_version_2.csv")
df_2021.to_csv(final_csv_path, index=False)

print(f"Les prédictions ont été sauvegardées dans : {final_csv_path}")


In [ ]:
import pandas as pd
import numpy as np
import os
import time
import psutil
import gc
from joblib import Parallel, delayed
from scipy.stats import norm

# Démarrer le chronomètre
start_time = time.time()

# Définition des chemins
input_csv_2021 = r"D:\wealth_predict_sentinel\Data\processed_csv\fullyConv_sans_augm_couche_gelee_batch_16_test_with_poverty_2021_logistic_reg_version_2.csv"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"

# Chargement des résultats de 2021
df_2021 = pd.read_csv(input_csv_2021)

# Vérification des colonnes nécessaires
required_columns = ['hhweight', 'hhsize', 'pcexp_binaire_pred']

missing_columns = [col for col in required_columns if col not in df_2021.columns]
if missing_columns:
    raise ValueError(f"Les colonnes suivantes sont manquantes dans les données de 2021 : {missing_columns}")

# Nombre total d'itérations bootstrap
n_bootstrap = 1000
batch_size = 100  # Taille du batch pour optimiser la mémoire
n_cores = 12  # Nombre de cœurs à utiliser pour le traitement parallèle

# Fonction pour exécuter une itération bootstrap
def bootstrap_iteration(seed, df):
    """Effectue une itération bootstrap et retourne le taux de pauvreté"""
    np.random.seed(seed)
    sample_indices = np.random.choice(df.index, size=len(df), replace=True)
    sample_df = df.loc[sample_indices]

    # Calcul du taux de pauvreté pondéré pour cet échantillon bootstrap
    weighted_poverty = (
        (sample_df['hhweight'] * sample_df['hhsize'] * sample_df['pcexp_binaire_pred']).sum() /
        (sample_df['hhweight'] * sample_df['hhsize']).sum()
    )
    return weighted_poverty

# Liste pour stocker les résultats bootstrap
bootstrap_poverty_rates = []

# Découpage en batches pour optimiser la mémoire
n_batches = (n_bootstrap + batch_size - 1) // batch_size

for batch in range(n_batches):
    start_idx = batch * batch_size
    end_idx = min((batch + 1) * batch_size, n_bootstrap)

    # Vérification de la mémoire disponible
    available_memory = psutil.virtual_memory().available / (1024 * 1024 * 1024)  # en Go
    print(f"Mémoire disponible: {available_memory:.2f} GB")

    # Pause si la mémoire devient insuffisante
    if available_memory < 2:
        print("⚠ Mémoire faible, pause de 10 secondes pour libérer de l'espace...")
        time.sleep(10)
        gc.collect()

    # Exécution parallèle du bootstrap
    seeds = range(start_idx, end_idx)
    batch_results = Parallel(n_jobs=n_cores)(
        delayed(bootstrap_iteration)(seed, df_2021) for seed in seeds
    )

    # Ajout des résultats
    bootstrap_poverty_rates.extend(batch_results)

    # Libération de la mémoire
    gc.collect()
    print(f"Batch {batch + 1}/{n_batches} terminé. Résultats accumulés: {len(bootstrap_poverty_rates)}")

# Conversion en tableau NumPy
bootstrap_poverty_rates = np.array(bootstrap_poverty_rates)

# Calcul des statistiques
mean_poverty = np.mean(bootstrap_poverty_rates)
std_poverty = np.std(bootstrap_poverty_rates, ddof=1)

# Intervalle de confiance à 95%
ci_lower = np.percentile(bootstrap_poverty_rates, 2.5)
ci_upper = np.percentile(bootstrap_poverty_rates, 97.5)

# Affichage des résultats
print(f"\n📊 Résumé des résultats:")
print(f"Taux de pauvreté prévisionnel moyen en 2021 : {mean_poverty:.4%}")
print(f"Intervalle de confiance à 95% : [{ci_lower:.4%}, {ci_upper:.4%}]")
print(f"Erreur standard : {std_poverty:.4%}")

# Sauvegarde des résultats dans un fichier CSV
bootstrap_results_path = os.path.join(output_dir_csv, "bootstrap_poverty_2021_results_gelee_version_2.csv")
pd.DataFrame({"bootstrap_poverty_rate": bootstrap_poverty_rates}).to_csv(bootstrap_results_path, index=False)

print(f"\n✅ Les résultats bootstrap ont été sauvegardés dans : {bootstrap_results_path}")

# Mesurer le temps d'exécution total
end_time = time.time()
elapsed_time = end_time - start_time

# Affichage du temps d'exécution
print("\n🔹 Temps total d'exécution :")
print(f"⏳ {elapsed_time:.2f} secondes")
print(f"⏳ {elapsed_time / 60:.2f} minutes")


### Essai avec le modèle basé sur le regular split

In [ ]:

import pandas as pd
import os
import pickle

# Définition des chemins
input_csv_2021 = r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConv_sans_augm_couche_gelee_batch_16_base_EHCVM_2018_pour_an_2021_version_2.csv" #Pour 2023 remplacer ce fichier
model_path = r"D:\wealth_predict_sentinel\models\fullyConv_sans_augm_couche_gelee_batch_16_logistic_pca_version_2.pkl"
scaler_path = r"D:\wealth_predict_sentinel\models\scaler_fullyConv_sans_augm_couche_gelee_batch_16_regular_split_version_2.pkl"
pca_path = r"D:\wealth_predict_sentinel\models\pca_fullyConv_sans_augm_couche_gelee_batch_16_regular_split_version_2.pkl"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"

# Chargement des données de 2021
df_2021 = pd.read_csv(input_csv_2021)

# Vérification des colonnes nécessaires
features = [f'feature_{i}' for i in range(4096)]
required_columns = features + ['hhweight', 'hhsize']  # Ajout des colonnes nécessaires au calcul du taux de pauvreté

missing_columns = [col for col in required_columns if col not in df_2021.columns]
if missing_columns:
    raise ValueError(f"Les colonnes suivantes sont manquantes dans les données de 2021 : {missing_columns}")

X_2021 = df_2021[features]

# Chargement du scaler et du PCA utilisés en 2018
with open(scaler_path, 'rb') as file:
    scaler = pickle.load(file)

with open(pca_path, 'rb') as file:
    pca = pickle.load(file)

# Normalisation des données 2021 avec le même scaler
X_2021_scaled = scaler.transform(X_2021)

# Réduction de dimension avec le même PCA
X_2021_pca = pca.transform(X_2021_scaled)

# Chargement du modèle entraîné en 2018
# La ligne suivante charge le modèle entraîné avec ses hyperparamètres, y compris class_weight='balanced'
with open(model_path, 'rb') as file:
    logistic_model = pickle.load(file)

# Prédictions sur les données de 2021
y_pred_2021 = logistic_model.predict(X_2021_pca)

# Ajout des prédictions au DataFrame
df_2021['pcexp_binaire_pred'] = y_pred_2021

# Calcul du taux de pauvreté prévisionnel pour 2021 en utilisant les pondérations de 2018
weighted_predicted_poverty_2021 = (
    (df_2021['hhweight'] * df_2021['hhsize'] * df_2021['pcexp_binaire_pred']).sum() /
    (df_2021['hhweight'] * df_2021['hhsize']).sum()
)

print(f"Taux de pauvreté prévisionnel en 2021 : {weighted_predicted_poverty_2021:.4%}")

# Sauvegarde des résultats finaux
final_csv_path = os.path.join(output_dir_csv, "fullyConv_sans_augm_couche_gelee_batch_16_test_with_poverty_2021_logistic_reg_regular_split_version_2.csv")
df_2021.to_csv(final_csv_path, index=False)

print(f"Les prédictions ont été sauvegardées dans : {final_csv_path}")


In [ ]:

import pandas as pd
import numpy as np
import os
import time
import psutil
import gc
from joblib import Parallel, delayed
from scipy.stats import norm

# Démarrer le chronomètre
start_time = time.time()

# Définition des chemins
input_csv_2021 = r"D:\wealth_predict_sentinel\Data\processed_csv\fullyConv_sans_augm_couche_gelee_batch_16_test_with_poverty_2021_logistic_reg_regular_split_version_2.csv"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"

# Chargement des résultats de 2021
df_2021 = pd.read_csv(input_csv_2021)

# Vérification des colonnes nécessaires
required_columns = ['hhweight', 'hhsize', 'pcexp_binaire_pred']

missing_columns = [col for col in required_columns if col not in df_2021.columns]
if missing_columns:
    raise ValueError(f"Les colonnes suivantes sont manquantes dans les données de 2021 : {missing_columns}")

# Nombre total d'itérations bootstrap
n_bootstrap = 1000
batch_size = 100  # Taille du batch pour optimiser la mémoire
n_cores = 12  # Nombre de cœurs à utiliser pour le traitement parallèle

# Fonction pour exécuter une itération bootstrap
def bootstrap_iteration(seed, df):
    """Effectue une itération bootstrap et retourne le taux de pauvreté"""
    np.random.seed(seed)
    sample_indices = np.random.choice(df.index, size=len(df), replace=True)
    sample_df = df.loc[sample_indices]

    # Calcul du taux de pauvreté pondéré pour cet échantillon bootstrap
    weighted_poverty = (
        (sample_df['hhweight'] * sample_df['hhsize'] * sample_df['pcexp_binaire_pred']).sum() /
        (sample_df['hhweight'] * sample_df['hhsize']).sum()
    )
    return weighted_poverty

# Liste pour stocker les résultats bootstrap
bootstrap_poverty_rates = []

# Découpage en batches pour optimiser la mémoire
n_batches = (n_bootstrap + batch_size - 1) // batch_size

for batch in range(n_batches):
    start_idx = batch * batch_size
    end_idx = min((batch + 1) * batch_size, n_bootstrap)

    # Vérification de la mémoire disponible
    available_memory = psutil.virtual_memory().available / (1024 * 1024 * 1024)  # en Go
    print(f"Mémoire disponible: {available_memory:.2f} GB")

    # Pause si la mémoire devient insuffisante
    if available_memory < 2:
        print("⚠ Mémoire faible, pause de 10 secondes pour libérer de l'espace...")
        time.sleep(10)
        gc.collect()

    # Exécution parallèle du bootstrap
    seeds = range(start_idx, end_idx)
    batch_results = Parallel(n_jobs=n_cores)(
        delayed(bootstrap_iteration)(seed, df_2021) for seed in seeds
    )

    # Ajout des résultats
    bootstrap_poverty_rates.extend(batch_results)

    # Libération de la mémoire
    gc.collect()
    print(f"Batch {batch + 1}/{n_batches} terminé. Résultats accumulés: {len(bootstrap_poverty_rates)}")

# Conversion en tableau NumPy
bootstrap_poverty_rates = np.array(bootstrap_poverty_rates)

# Calcul des statistiques
mean_poverty = np.mean(bootstrap_poverty_rates)
std_poverty = np.std(bootstrap_poverty_rates, ddof=1)

# Intervalle de confiance à 95%
ci_lower = np.percentile(bootstrap_poverty_rates, 2.5)
ci_upper = np.percentile(bootstrap_poverty_rates, 97.5)

# Affichage des résultats
print(f"\n📊 Résumé des résultats:")
print(f"Taux de pauvreté prévisionnel moyen en 2021 : {mean_poverty:.4%}")
print(f"Intervalle de confiance à 95% : [{ci_lower:.4%}, {ci_upper:.4%}]")
print(f"Erreur standard : {std_poverty:.4%}")

# Sauvegarde des résultats dans un fichier CSV
bootstrap_results_path = os.path.join(output_dir_csv, "bootstrap_poverty_2021_results_regular_split_gelee_version_2.csv")
pd.DataFrame({"bootstrap_poverty_rate": bootstrap_poverty_rates}).to_csv(bootstrap_results_path, index=False)

print(f"\n✅ Les résultats bootstrap ont été sauvegardés dans : {bootstrap_results_path}")

# Mesurer le temps d'exécution total
end_time = time.time()
elapsed_time = end_time - start_time

# Affichage du temps d'exécution
print("\n🔹 Temps total d'exécution :")
print(f"⏳ {elapsed_time:.2f} secondes")
print(f"⏳ {elapsed_time / 60:.2f} minutes")
